In [1]:
import pandas as pd

from src.config.settings import (
    DATA_DIR,
    SELECTED_FEATURES,
    TARGET_COLUMN,
    RAW_INT_FEATURES,
    RAW_FLOAT_FEATURES,
    RAW_CATEGORICAL_FEATURES,
)

from src.features.select_features import select_model_features
from src.features.type_features import cast_feature_types
from src.features.missing_values import clean_missing_values
from src.features.apply_one_hot_encoding import apply_one_hot_encoding
from src.features.feature_engineering import apply_feature_engineering

from src.inference.feature_contract import (
    build_raw_inference_feature_names,
    ensure_feature_engineering_columns,
    align_to_model_feature_contract,
)

from src.inference.predict import load_model_artifacts


In [2]:
artifacts = load_model_artifacts()

model = artifacts.model
scaler = artifacts.scaler
feature_names = artifacts.feature_names
device = artifacts.device

print("Modelo carregado:", model is not None)
print("Scaler carregado:", scaler is not None)
print("Qtd features do modelo:", len(feature_names))
print(feature_names)

Modelo carregado: True
Scaler carregado: True
Qtd features do modelo: 37
['Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'Gender_Male', 'Senior Citizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'Phone Service_Yes', 'Multiple Lines_No phone service', 'Multiple Lines_Yes', 'Internet Service_Fiber optic', 'Internet Service_No', 'Online Security_No internet service', 'Online Security_Yes', 'Online Backup_No internet service', 'Online Backup_Yes', 'Device Protection_No internet service', 'Device Protection_Yes', 'Tech Support_No internet service', 'Tech Support_Yes', 'Streaming TV_No internet service', 'Streaming TV_Yes', 'Streaming Movies_No internet service', 'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year', 'Paperless Billing_Yes', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check', 'total_services', 'fiber_price_impact', 'avg_ticket', 'is_new_customer']


In [ ]:
csv_path = DATA_DIR / "raw" / "telco_customer_churn.csv"

df_full = pd.read_csv(csv_path, sep=";")

print(df_full.shape)
print(df_full.columns.tolist())
df_full.head()

(7043, 1)
['CustomerID;Count;Country;State;City;Zip Code;Lat Long;Latitude;Longitude;Gender;Senior Citizen;Partner;Dependents;Tenure Months;Phone Service;Multiple Lines;Internet Service;Online Security;Online Backup;Device Protection;Tech Support;Streaming TV;Streaming Movies;Contract;Paperless Billing;Payment Method;Monthly Charges;Total Charges;Churn Label;Churn Value;Churn Score;CLTV;Churn Reason']


,CustomerID;Count;Country;State;City;Zip Code;Lat Long;Latitude;Longitude;Gender;Senior Citizen;Partner;Dependents;Tenure Months;Phone Service;Multiple Lines;Internet Service;Online Security;Online Backup;Device Protection;Tech Support;Streaming TV;Streaming Movies;Contract;Paperless Billing;Payment Method;Monthly Charges;Total Charges;Churn Label;Churn Value;Churn Score;CLTV;Churn Reason
3668-QPYBK;1;United States;California;Los Angeles;90003;33.964131,-118.272783;33.964131;-118.272783;Male;No;No;...
9237-HQITU;1;United States;California;Los Angeles;90005;34.059281,-118.30742;34.059281;-118.30742;Female;No;No;...
9305-CDSKC;1;United States;California;Los Angeles;90006;34.048013,-118.293953;34.048013;-118.293953;Female;No;N...
7892-POOKP;1;United States;California;Los Angeles;90010;34.062125,-118.315709;34.062125;-118.315709;Female;No;Y...
0280-XJGEX;1;United States;California;Los Angeles;90015;34.039224,-118.266293;34.039224;-118.266293;Male;No;No;...


In [4]:
raw_cols_to_check = [
    "Contract",
    "Internet Service",
    "Payment Method",
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
]

[col for col in raw_cols_to_check if col in df_full.columns]

[]

In [ ]:
RAW_INFERENCE_FEATURES = build_raw_inference_feature_names(
    selected_features=SELECTED_FEATURES,
    target_column=TARGET_COLUMN,
)

RAW_INFERENCE_INT_FEATURES = [
    feature for feature in RAW_INT_FEATURES if feature != TARGET_COLUMN
]

print("RAW_INFERENCE_FEATURES:")
print(RAW_INFERENCE_FEATURES)

print("RAW_INFERENCE_INT_FEATURES:")
print(RAW_INFERENCE_INT_FEATURES)

In [ ]:
sample = df_full.sample(1, random_state=42)

if TARGET_COLUMN in sample.columns:
    print("Target real:", sample[TARGET_COLUMN].iloc[0])

df = sample[RAW_INFERENCE_FEATURES].copy()

print("RAW")
print(df.T)

In [ ]:
df_selected_features = select_model_features(
    df,
    selected_features=RAW_INFERENCE_FEATURES,
)

print("Após select_model_features")
print(df_selected_features.T)

In [ ]:
df_typed_features = cast_feature_types(
    df_selected_features,
    int_features=RAW_INFERENCE_INT_FEATURES,
    float_features=RAW_FLOAT_FEATURES,
    categorical_features=RAW_CATEGORICAL_FEATURES,
)

print("Após cast_feature_types")
print(df_typed_features.T)
print(df_typed_features.dtypes)

In [ ]:
df_no_missing_values = clean_missing_values(
    df_typed_features,
    selected_features=RAW_INFERENCE_FEATURES,
    int_features=RAW_INFERENCE_INT_FEATURES,
    float_features=RAW_FLOAT_FEATURES,
    categorical_features=RAW_CATEGORICAL_FEATURES,
)

print("Após clean_missing_values")
print(df_no_missing_values.T)

In [ ]:
df_ohe = apply_one_hot_encoding(df_no_missing_values)

print("Após apply_one_hot_encoding")
print(df_ohe.T)
print(df_ohe.columns.tolist())

In [ ]:
df_ohe_ready_for_feat_eng = ensure_feature_engineering_columns(
    df_encoded=df_ohe,
    df_raw_features=df_no_missing_values,
)

print("Após ensure_feature_engineering_columns")
print(df_ohe_ready_for_feat_eng.T)
print(df_ohe_ready_for_feat_eng.columns.tolist())

In [ ]:
required_for_feature_engineering = [
    "Tenure Months",
    "Total Charges",
    "Monthly Charges",
    "Internet Service_Fiber optic",
    "Online Security_Yes",
    "Online Backup_Yes",
    "Device Protection_Yes",
    "Tech Support_Yes",
    "Streaming TV_Yes",
    "Streaming Movies_Yes",
]

missing = [
    col for col in required_for_feature_engineering
    if col not in df_ohe_ready_for_feat_eng.columns
]

print("Colunas ausentes antes do feature engineering:")
print(missing)

In [ ]:
df_feat_eng = apply_feature_engineering(df_ohe_ready_for_feat_eng)

print("Após apply_feature_engineering")
print(df_feat_eng.T)
print(df_feat_eng.columns.tolist())

In [ ]:
df_model_ready = align_to_model_feature_contract(
    df_feat_eng,
    model_feature_names=feature_names,
)

print("Após align_to_model_feature_contract")
print(df_model_ready.T)
print(df_model_ready.shape)

In [ ]:
print("RAW")
print(df.T)

print("Após select_model_features")
print(df_selected_features.T)

print("Após cast_feature_types")
print(df_typed_features.T)

print("Após clean_missing_values")
print(df_no_missing_values.T)

print("Após apply_one_hot_encoding")
print(df_ohe.T)

print("Após ensure_feature_engineering_columns")
print(df_ohe_ready_for_feat_eng.T)

print("Após apply_feature_engineering")
print(df_feat_eng.T)

print("Após align_to_model_feature_contract")
print(df_model_ready.T)